# Neural Network Parameter Counter
This notebook shows how to **automatically count the number of trainable parameters** (weights, biases, etc.) from C++ source files that implement various neural network architectures for embedded systems (e.g., Arduino).

## Why count parameters?
- **Memory estimation**: Each `float` parameter typically occupies 4 bytes. Knowing the parameter count helps estimate RAM/Flash usage.
- **Model comparison**: Different architectures (CNN, MLP, KAN, RBFN) have different parameter footprints.
- **Debugging**: Verify that the exported model matches the expected number of parameters.

## What do we parse?
We look for C++ array definitions like:
```cpp
const float weights[] PROGMEM = {0.1, -0.2, 0.3};
```
Each number inside the braces counts as one parameter.

We will also handle architecture‑specific patterns (e.g., KAN coefficients expressed in formulas).

---

In [ ]:
import re
import os
from typing import Dict, Tuple, List, Any

## 1. Convolutional Neural Network (CNN)

CNNs store parameters in **flat arrays** (convolutional kernels, dense layers, biases). The counting function:
- Scans for arrays declared with `const float ... PROGMEM = { ... }`
- Counts all floating‑point literals inside the braces.

We also show an optional **header‑based estimation** that reads `#define` constants to compute the expected number of parameters (useful for validation).

In [ ]:
def count_parameters_cpp(file_path: str) -> Tuple[int, Dict[str, int]]:
    """
    Count parameters from a .cpp file containing float arrays with PROGMEM.
    
    Returns:
        total_params: total number of float literals found
        layer_details: dict mapping array names to parameter counts
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Match arrays like: const float layer0[] PROGMEM = { 0.1, -0.2, ... }
    pattern = re.compile(
        r'const float\s+(\w+)\s*\[\]\s+PROGMEM\s*=\s*\{([^}]*)\}',
        re.DOTALL
    )

    matches = pattern.findall(content)

    total_params = 0
    layer_details = {}

    for array_name, values_block in matches:
        # Extract all floating-point numbers (with optional 'f' suffix)
        numbers = re.findall(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?f?', values_block)
        count = len(numbers)

        layer_details[array_name] = count
        total_params += count

    return total_params, layer_details


def estimate_from_header(file_path: str) -> Dict[str, int]:
    """
    Optional: read #define constants to compute expected parameter counts.
    This serves as a cross-check for the direct counting method.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    params = {}

    # Example: Dense layer defined by INPUT_SIZE and OUTPUT_SIZE
    dense_match = re.search(
        r'#define\s+DENSE0_INPUT_SIZE\s+(\d+).*?#define\s+DENSE0_OUTPUT_SIZE\s+(\d+)',
        content, re.DOTALL
    )
    if dense_match:
        in_size = int(dense_match.group(1))
        out_size = int(dense_match.group(2))
        params["dense0_weights"] = in_size * out_size
        params["dense0_biases"] = out_size
        params["dense0_total"] = params["dense0_weights"] + params["dense0_biases"]

    return params


# Example usage (commented because files may not exist)
# total, details = count_parameters_cpp("neural_network.cpp")
# print(total, details)

## 2. Fourier Analysis Network (FAN)

FANs use trigonometric expansions. The parameters are stored in **plain float arrays** (sometimes without `PROGMEM`). The same counting logic applies, but we also capture arrays that may be declared without the `const` keyword.

In [ ]:
def count_parameters_fan(file_path: str) -> Tuple[int, Dict[str, int]]:
    """
    Count parameters from a FAN header/source file.
    Looks for any float array definition (with or without 'const').
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Match: [const] float name[...] = { ... };
    pattern = re.compile(
        r'(?:const\s+)?float\s+(\w+)\s*\[.*?\]\s*=\s*\{([^}]*)\}',
        re.DOTALL
    )

    matches = pattern.findall(content)

    total = 0
    details = {}
    for name, block in matches:
        numbers = re.findall(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?f?', block)
        count = len(numbers)
        details[name] = count
        total += count

    return total, details


# Example:
# total, details = count_parameters_fan("fan_model.h")
# print(total, details)

## 3. Multilayer Perceptron (MLP)

MLPs store weights and biases in arrays. The code is similar to the FAN counter but uses a slightly more flexible regex to capture multi‑line initializations. We also handle scientific notation (e.g., `1.2e-5`).

In [ ]:
def count_mlp_parameters(file_path: str) -> Tuple[int, Dict[str, int]]:
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Captures arrays even if they span multiple lines
    pattern = re.compile(
        r'(?:const\s+)?float\s+(\w+)\s*\[.*?\]\s*=\s*\{([\s\S]*?)\};',
        re.DOTALL
    )

    total = 0
    details = {}
    for name, block in pattern.findall(content):
        # Match any valid floating-point literal
        numbers = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?f?', block)
        count = len(numbers)
        details[name] = count
        total += count

    return total, details


# Example:
# total, details = count_mlp_parameters("MLP.h")
# print(total, details)

## 4. Kolmogorov–Arnold Network (KAN)

KANs are different: parameters appear **inside mathematical expressions** rather than in explicit arrays. For example:
```cpp
output = 0.5 * x_0 + 0.3 * std::pow(x_1, 2) - 0.1;
```
We count:
- **Linear terms**: coefficients multiplied by `x_i`
- **Power terms**: coefficients multiplied by `std::pow(...)`
- **Bias‑like constants**: standalone numeric literals added or subtracted

This requires a completely different parsing strategy based on regular expressions.

In [ ]:
def count_parameters_kan(header_path: str) -> int:
    """
    Count parameters from a KAN implementation where parameters are embedded in formulas.
    Returns total number of coefficient occurrences.
    """
    with open(header_path, 'r', encoding='utf-8') as f:
        content = f.read()

    float_pattern = r'[-+]?\d*\.\d+(?:e[-+]?\d+)?'

    # 1) Coefficients before * x_i
    linear = re.findall(rf'({float_pattern})\s*\*\s*x_\d+', content)

    # 2) Coefficients before * std::pow(...)
    pow_terms = re.findall(rf'({float_pattern})\s*\*\s*std::pow', content)

    # 3) Standalone constants (bias) that are not part of a multiplication
    #    We look for +- constant (but avoid matching inside a multiplication)
    bias = re.findall(rf'[\+\-]\s*({float_pattern})(?!\s*\*)', content)

    total = len(linear) + len(pow_terms) + len(bias)

    print("=== KAN Parameter Count ===")
    print(f"Linear coefficients : {len(linear)}")
    print(f"Pow coefficients     : {len(pow_terms)}")
    print(f"Bias constants       : {len(bias)}")
    print(f"TOTAL                : {total}")

    return total


# Example:
# count_parameters_kan("kan_model.h")

## 5. Radial Basis Function Network (RBFN)

RBFNs have three types of parameters:
- **Centers** (often named `center` or `mu`)
- **Widths** (often named `sigma` or `gamma`)
- **Weights** (linear output layer)
- **Biases**

We combine **multiple files** (`.h` and `.cpp`) and classify each array by its name to provide a breakdown.

In [ ]:
def read_multiple_files(file_paths: List[str]) -> str:
    """Concatenate content of several files."""
    content = ""
    loaded = []
    for path in file_paths:
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                content += f.read() + "\n"
                loaded.append(path)
        else:
            print(f"[Warning] File not found: {path}")
    if not loaded:
        raise FileNotFoundError("No valid files provided.")
    print(f"Loaded files: {loaded}")
    return content


def classify_rbf_parameter(name: str) -> str:
    """Heuristic to guess the role of a parameter array."""
    name_low = name.lower()
    if "bias" in name_low or name_low.startswith("b"):
        return "bias"
    if "weight" in name_low or name_low.startswith("w"):
        return "weight"
    if "center" in name_low or "mu" in name_low:
        return "rbf_center"
    if "sigma" in name_low or "gamma" in name_low or "width" in name_low:
        return "rbf_width"
    return "unknown"


def count_parameters_rbfn(file_paths: List[str]) -> Tuple[int, Dict[str, int], Dict[str, Any]]:
    """
    Count and classify RBFN parameters from one or more files.
    Returns:
        total: total parameter count
        summary: dict with counts per type (weight, bias, rbf_center, rbf_width, unknown)
        details: dict mapping array names to count and type
    """
    content = read_multiple_files(file_paths)

    pattern = re.compile(
        r'(?:const\s+)?float\s+(\w+)\s*\[.*?\]\s*(?:PROGMEM)?\s*=\s*\{([\s\S]*?)\};',
        re.DOTALL
    )

    total = 0
    summary = {
        "weight": 0,
        "bias": 0,
        "rbf_center": 0,
        "rbf_width": 0,
        "unknown": 0
    }
    details = {}

    for name, block in pattern.findall(content):
        numbers = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?f?', block)
        count = len(numbers)
        ptype = classify_rbf_parameter(name)

        details[name] = {"count": count, "type": ptype}
        total += count
        summary[ptype] += count

    return total, summary, details


# Example:
# files = ["RbfModelRBF.h", "RbfModelRBF.cpp"]
# total, summary, details = count_parameters_rbfn(files)
# print(total, summary)